# Procesamiento de Datos a Gran Escala — Notebook de Clase
### Pontificia Universidad Javeriana
### Profesor: Fabián Alexis Pallares Jaimes

---

**Dataset:** Datos de empleados de una empresa (clientes potenciales de un producto financiero).  
**Historia:** Somos el equipo de analítica de RR.HH. Queremos entender a nuestros empleados, predecir quién abandona la empresa, y segmentar grupos con características similares.

Este notebook cubre, en el ciclo de **CRISP-DM**, desde procesamiento de datos en Spark:

| Sección | Tema |
|---|---|
| 1 | Preprocesamiento con Spark SQL + MLlib Feature Engineering |
| 2 | Aprendizaje Supervisado — Clasificación y Regresión |
| 3 | Aprendizaje No Supervisado — Clustering K-Means |
| 4 | Métricas de Rendimiento — Evaluación de todos los modelos |

---
## Sección 0 — Configuración e Inicio de SparkSession

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# ── Rutas del entorno local WSL ──────────────────────────────────────────────
SPARK_HOME = os.environ.get('SPARK_HOME', '/opt/spark')  # o /usr/local/spark
HDFS_URI   = 'hdfs://localhost:9000'
HDFS_DIR   = f'{HDFS_URI}/datasets/empleados'

os.environ['PYSPARK_PYTHON']        = 'python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

print(f'SPARK_HOME  : {SPARK_HOME}')
print(f'HDFS_URI    : {HDFS_URI}')
print(f'HDFS_DIR    : {HDFS_DIR}')

SPARK_HOME  : /home/fpallares/spark
HDFS_URI    : hdfs://localhost:9000
HDFS_DIR    : hdfs://localhost:9000/datasets/empleados


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('PDGE-Clase-Completa')
    .master('spark://localhost:7077')
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory', '2g')
    .config('spark.executor.cores', '2')
    # HDFS como sistema de archivos por defecto
    .config('spark.hadoop.fs.defaultFS', HDFS_URI)
    # Evitar warnings de serialización
    .config('spark.sql.repl.eagerEval.enabled', 'true')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

print('   SparkSession lista')
print(f'   Versión Spark : {spark.version}')
print(f'   App Name      : {spark.sparkContext.appName}')
print(f'   Master        : {spark.sparkContext.master}')
print()
print('   Spark UI disponible en: http://localhost:4040')

26/05/06 11:17:42 WARN Utils: Your hostname, BOGOFI38A resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 11:17:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 11:17:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


   SparkSession lista
   Versión Spark : 3.5.2
   App Name      : PDGE-Clase-Completa
   Master        : spark://localhost:7077

   Spark UI disponible en: http://localhost:4040


> **💡 Spark UI:** Mientras este notebook corre, puedes abrir `http://localhost:4040` en tu navegador y ver en tiempo real los Jobs, Stages y Tasks que Spark ejecuta.

---
## Sección 1 — Preprocesamiento de Datos

**Objetivo:** Tomar datos brutos con problemas de calidad y producir la **vista minable**: un DataFrame con columna `features` (Vector) y columna `label` (Double), listo para cualquier modelo de ML.

### 1.1 Crear el dataset y guardarlo en HDFS

In [3]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType
)

# Dataset de empleados — 20 filas con problemas de calidad intencionales
# Columnas: id, edad, salario, antiguedad, dept, satisfaccion, abandono (target)
schema = StructType([
    StructField('id',           IntegerType(), True),
    StructField('edad',         IntegerType(), True),   # tiene nulos
    StructField('salario',      DoubleType(),  True),   # tiene outlier y nulos
    StructField('antiguedad',   IntegerType(), True),
    StructField('dept',         StringType(),  True),   # categórico, tiene nulos
    StructField('satisfaccion', DoubleType(),  True),   # escala 1.0-5.0
    StructField('abandono',     IntegerType(), True),   # target: 0=se queda, 1=abandona
])

data_raw = [
    # id   edad  salario    antig  dept        satisf  abandono
    (1,    28,   3500.0,    3,     'IT',       4.2,    0),
    (2,    34,   4200.0,    8,     'Ventas',   3.1,    1),
    (3,    None, 3100.0,    2,     'IT',       3.8,    0),   # edad nula
    (4,    45,   None,      12,    'RRHH',     2.5,    1),   # salario nulo
    (5,    29,   3800.0,    5,     'Ventas',   4.0,    0),
    (6,    52,   180000.0,  15,    'IT',       1.8,    1),   # salario outlier
    (7,    38,   4100.0,    9,     None,       3.5,    0),   # dept nulo
    (8,    28,   3500.0,    3,     'IT',       4.2,    0),   # duplicado de fila 1
    (9,    61,   5200.0,    20,    'RRHH',     2.0,    1),
    (10,   33,   3900.0,    7,     'Ventas',   4.5,    0),
    (11,   41,   4800.0,    11,    'IT',       3.9,    0),
    (12,   27,   3200.0,    1,     'Ventas',   2.8,    1),
    (13,   55,   6100.0,    18,    'RRHH',     3.2,    0),
    (14,   36,   4300.0,    8,     'IT',       4.7,    0),
    (15,   48,   5500.0,    14,    'Ventas',   1.5,    1),
    (16,   31,   3700.0,    4,     'RRHH',     4.1,    0),
    (17,   None, 4000.0,    6,     'IT',       3.6,    0),   # edad nula
    (18,   44,   4600.0,    10,    'Ventas',   2.3,    1),
    (19,   39,   4400.0,    9,     'RRHH',     3.8,    0),
    (20,   26,   3300.0,    2,     'IT',       4.4,    0),
]

df_raw = spark.createDataFrame(data_raw, schema)

# ── Guardar en HDFS (formato Parquet — columnar, eficiente) ──────────────────
df_raw.write.mode('overwrite').parquet(f'{HDFS_DIR}/empleados_raw.parquet')
print(f'Dataset guardado en HDFS: {HDFS_DIR}/empleados_raw.parquet')
print(f'Filas: {df_raw.count()} · Columnas: {len(df_raw.columns)}')

Dataset guardado en HDFS: hdfs://localhost:9000/datasets/empleados/empleados_raw.parquet
Filas: 20 · Columnas: 7


### 1.2 Leer desde HDFS y explorar

In [4]:
# Leer desde HDFS
df = spark.read.parquet(f'{HDFS_DIR}/empleados_raw.parquet')

print('── Schema ──────────────────────────────────────────')
df.printSchema()

print('── Primeras 5 filas ────────────────────────────────')
df.show(5)

── Schema ──────────────────────────────────────────
root
 |-- id: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- salario: double (nullable = true)
 |-- antiguedad: integer (nullable = true)
 |-- dept: string (nullable = true)
 |-- satisfaccion: double (nullable = true)
 |-- abandono: integer (nullable = true)

── Primeras 5 filas ────────────────────────────────
+---+----+-------+----------+------+------------+--------+
| id|edad|salario|antiguedad|  dept|satisfaccion|abandono|
+---+----+-------+----------+------+------------+--------+
| 11|  41| 4800.0|        11|    IT|         3.9|       0|
| 12|  27| 3200.0|         1|Ventas|         2.8|       1|
| 13|  55| 6100.0|        18|  RRHH|         3.2|       0|
| 14|  36| 4300.0|         8|    IT|         4.7|       0|
| 15|  48| 5500.0|        14|Ventas|         1.5|       1|
+---+----+-------+----------+------+------------+--------+
only showing top 5 rows



In [5]:
# Estadísticas descriptivas de columnas numéricas
print('── Estadísticas descriptivas ───────────────────────')
df.describe(['edad', 'salario', 'antiguedad', 'satisfaccion']).show()

── Estadísticas descriptivas ───────────────────────
+-------+------------------+------------------+------------------+------------------+
|summary|              edad|           salario|        antiguedad|      satisfaccion|
+-------+------------------+------------------+------------------+------------------+
|  count|                18|                19|                20|                20|
|   mean|38.611111111111114|13431.578947368422|              8.35|3.3950000000000005|
| stddev| 10.41005672634648| 40344.24167408913|5.4219340111304035|0.9539254080956888|
|    min|                26|            3100.0|                 1|               1.5|
|    max|                61|          180000.0|                20|               4.7|
+-------+------------------+------------------+------------------+------------------+



In [6]:
from pyspark.sql.functions import col, when, count, isnan

# Contar nulos por columna
print('── Conteo de nulos por columna ─────────────────────')
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

# Contar duplicados
total   = df.count()
unicos  = df.dropDuplicates().count()
print(f'Total filas: {total}  |  Únicas: {unicos}  |  Duplicadas: {total - unicos}')

── Conteo de nulos por columna ─────────────────────
+---+----+-------+----------+----+------------+--------+
| id|edad|salario|antiguedad|dept|satisfaccion|abandono|
+---+----+-------+----------+----+------------+--------+
|  0|   2|      1|         0|   1|           0|       0|
+---+----+-------+----------+----+------------+--------+

Total filas: 20  |  Únicas: 20  |  Duplicadas: 0


### 1.3 Limpieza con DataFrame API (Spark SQL)

In [7]:
from pyspark.sql import functions as F

# PASO 1 — Eliminar duplicados exactos
df_clean = df.dropDuplicates()
print(f'Después de dropDuplicates: {df_clean.count()} filas')

Después de dropDuplicates: 20 filas


In [8]:
# PASO 2 — Filtrar outlier de salario (> 3 desv. estándar)
stats       = df_clean.select(
    F.mean('salario').alias('media'),
    F.stddev('salario').alias('std')
).collect()[0]
limite_sup  = stats['media'] + 3 * stats['std']
df_clean    = df_clean.filter(col('salario') < limite_sup)
print(f'Después de filtrar outliers de salario (umbral={limite_sup:.0f}): {df_clean.count()} filas')

Después de filtrar outliers de salario (umbral=134464): 18 filas


In [9]:
# PASO 3 — Imputar nulos numéricos con la mediana (distribución asimétrica)
mediana_edad    = df_clean.approxQuantile('edad',    [0.5], 0.01)[0]
mediana_salario = df_clean.approxQuantile('salario', [0.5], 0.01)[0]
df_clean = df_clean.fillna({
    'edad':    int(mediana_edad),
    'salario': mediana_salario
})
print(f'Mediana edad imputada: {mediana_edad:.0f}  |  Mediana salario imputada: {mediana_salario:.0f}')

Mediana edad imputada: 34  |  Mediana salario imputada: 4000


In [10]:
# PASO 4 — Imputar nulos categóricos con moda
moda_dept = (
    df_clean.filter(col('dept').isNotNull())
    .groupBy('dept').count()
    .orderBy(F.desc('count'))
    .first()['dept']
)
df_clean = df_clean.fillna({'dept': moda_dept})
print(f'Moda de dept imputada: {moda_dept}')

Moda de dept imputada: IT


In [11]:
print(f'\nDataset limpio: {df_clean.count()} filas · {len(df_clean.columns)} columnas')


Dataset limpio: 18 filas · 7 columnas


In [12]:
# Verificar que no quedan nulos
print('── Nulos después de limpieza ───────────────────────')
df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).show()

print('── Dataset limpio ──────────────────────────────────')
df_clean.show()

── Nulos después de limpieza ───────────────────────
+---+----+-------+----------+----+------------+--------+
| id|edad|salario|antiguedad|dept|satisfaccion|abandono|
+---+----+-------+----------+----+------------+--------+
|  0|   0|      0|         0|   0|           0|       0|
+---+----+-------+----------+----+------------+--------+

── Dataset limpio ──────────────────────────────────
+---+----+-------+----------+------+------------+--------+
| id|edad|salario|antiguedad|  dept|satisfaccion|abandono|
+---+----+-------+----------+------+------------+--------+
|  2|  34| 4200.0|         8|Ventas|         3.1|       1|
|  5|  29| 3800.0|         5|Ventas|         4.0|       0|
|  9|  61| 5200.0|        20|  RRHH|         2.0|       1|
|  8|  28| 3500.0|         3|    IT|         4.2|       0|
| 10|  33| 3900.0|         7|Ventas|         4.5|       0|
|  1|  28| 3500.0|         3|    IT|         4.2|       0|
| 15|  48| 5500.0|        14|Ventas|         1.5|       1|
| 16|  31| 3700.0|

### 1.4 Análisis de correlación

In [13]:
num_cols = ['edad', 'salario', 'antiguedad', 'satisfaccion', 'abandono']

print('── Matriz de correlación de Pearson ────────────────')
print(f'{"":15}', end='')
for c in num_cols:
    print(f'{c:14}', end='')
print()
for c1 in num_cols:
    print(f'{c1:15}', end='')
    for c2 in num_cols:
        r = df_clean.stat.corr(c1, c2)
        print(f'{r:+.3f}         ', end='')
    print()

── Matriz de correlación de Pearson ────────────────
               edad          salario       antiguedad    satisfaccion  abandono      
edad           +1.000         +0.896         +0.966         -0.660         +0.379         
salario        +0.896         +1.000         +0.949         -0.558         +0.282         
antiguedad     +0.966         +0.949         +1.000         -0.587         +0.334         
satisfaccion   -0.660         -0.558         -0.587         +1.000         -0.851         
abandono       +0.379         +0.282         +0.334         -0.851         +1.000         


### 1.5 Ingeniería de características con Spark MLlib

> Aquí pasamos de **Spark SQL** a **Spark MLlib**. Las operaciones de abajo producen el formato que los modelos de ML requieren.

In [14]:
#!pip install numpy

In [15]:
import numpy as np

In [16]:
from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler,
    StandardScaler,
    MinMaxScaler,
    Bucketizer
)
from pyspark.ml import Pipeline

In [17]:
# PASO A — Codificar variable categórica 'dept' → índice numérico
# StringIndexer asigna 0.0 al valor más frecuente, 1.0 al siguiente, etc.
indexer = StringIndexer(
    inputCol='dept',
    outputCol='dept_idx',
    handleInvalid='keep'
)

In [18]:
# PASO B — Discretizar 'satisfaccion' en rangos
# 1.0-2.5 = insatisfecho(0), 2.5-3.5 = neutral(1), 3.5-5.0 = satisfecho(2)
bucketizer = Bucketizer(
    splits=[1.0, 2.5, 3.5, 5.1],
    inputCol='satisfaccion',
    outputCol='satisf_grupo'
)

In [19]:
# PASO C — Ensamblar todas las features numéricas en un vector
feature_cols = ['edad', 'salario', 'antiguedad', 'satisfaccion', 'dept_idx']
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features_raw',
    handleInvalid='keep'
)

In [20]:
# PASO D — Estandarizar (media=0, std=1)
# Fundamental para KMeans (sección 3) y modelos basados en distancias
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=True,
    withStd=True
)

In [21]:
# PASO E — Pipeline de preprocesamiento completo
# La columna 'abandono' es nuestro target (label)
prep_pipeline = Pipeline(stages=[indexer, bucketizer, assembler, scaler])

In [ ]:
df_train, df_test = df_clean.randomSplit([0.75, 0.25], seed=42)
print(f'Train: {df_train.count()} filas  |  Test: {df_test.count()} filas')

# Entrenar el pipeline
prep_model = prep_pipeline.fit(df_train)

# Transformar ambos conjuntos
train_prep = prep_model.transform(df_train)
test_prep  = prep_model.transform(df_test)

Train: 13 filas  |  Test: 5 filas


In [ ]:
# Renombrar 'abandono' a 'label' (convención de MLlib)
from pyspark.sql.functions import col as scol
train_prep = train_prep.withColumn('label', scol('abandono').cast('double'))
test_prep  = test_prep.withColumn('label',  scol('abandono').cast('double'))

print('\n── Vista minable (columnas relevantes) ─────────────')
train_prep.select('id', 'features', 'label').show(5, truncate=False)
print('Vista minable generada — features (Vector) + label (Double)')

In [ ]:
# Guardar las vistas minables en HDFS para reutilizarlas
train_prep.write.mode('overwrite').parquet(f'{HDFS_DIR}/train_prep.parquet')
test_prep.write.mode('overwrite').parquet(f'{HDFS_DIR}/test_prep.parquet')

print(f'Guardado en HDFS:')
print(f'{HDFS_DIR}/train_prep.parquet')
print(f'{HDFS_DIR}/test_prep.parquet')

---
## Sección 2 — Aprendizaje Supervisado

**Objetivo:** Predecir si un empleado va a **abandonar la empresa** (clasificación) y predecir su **salario esperado** (regresión).  
Usamos la vista minable generada en la sección anterior.

### 2.1 Clasificación — ¿El empleado abandonará? (Decision Tree)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression

# ── Árbol de Decisión ────────────────────────────────────────────────────────
# maxDepth controla la complejidad del árbol (más profundo = más complejo = más overfitting)
dt_clf = DecisionTreeClassifier(
    featuresCol='features',
    labelCol='label',
    maxDepth=3,         # profundidad máxima del árbol
    seed=42
)

# Entrenar
dt_model = dt_clf.fit(train_prep)

# Predecir sobre el test set
pred_dt = dt_model.transform(test_prep)

print('── Predicciones Árbol de Decisión ──────────────────')
pred_dt.select('id', 'label', 'prediction', 'probability').show(truncate=False)

In [ ]:
# ── Regresión Logística (segundo clasificador para comparar) ─────────────────
lr_clf = LogisticRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=20
)

lr_model  = lr_clf.fit(train_prep)
pred_lr   = lr_model.transform(test_prep)

print('── Predicciones Regresión Logística ────────────────')
pred_lr.select('id', 'label', 'prediction', 'probability').show(truncate=False)

### 2.2 Regresión — Predecir el salario del empleado

In [ ]:
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor

# Para regresión el 'label' es el salario (numérico continuo)
# Features: edad, antiguedad, satisfaccion, dept_idx (todo excepto salario)
feature_cols_reg = ['edad', 'antiguedad', 'satisfaccion', 'dept_idx']

In [ ]:
assembler_reg = VectorAssembler(
    inputCols=feature_cols_reg,
    outputCol='features_reg',
    handleInvalid='keep'
)

In [ ]:
# Reescalar para regresión
scaler_reg = StandardScaler(
    inputCol='features_reg',
    outputCol='features_scaled_reg',
    withMean=True, withStd=True
)

In [ ]:
# Pipeline de features para regresión
pipe_reg = Pipeline(stages=[indexer, assembler_reg, scaler_reg])

# Fit sobre train_clean y transform ambos conjuntos
pipe_reg_model = pipe_reg.fit(df_train)
train_reg = pipe_reg_model.transform(df_train).withColumn('label', scol('salario'))
test_reg  = pipe_reg_model.transform(df_test).withColumn('label',  scol('salario'))

print('── Features de regresión (sin salario) ─────────────')
train_reg.select('id', 'features_scaled_reg', 'label').show(5, truncate=False)

In [ ]:
# ── Regresión Lineal ─────────────────────────────────────────────────────────
lr_reg = LinearRegression(
    featuresCol='features_scaled_reg',
    labelCol='label',
    maxIter=20,
    regParam=0.1    # regularización para evitar overfitting
)

lr_reg_model = lr_reg.fit(train_reg)
pred_reg     = lr_reg_model.transform(test_reg)

print('── Predicciones de Salario ─────────────────────────')
pred_reg.select(
    'id',
    scol('label').alias('salario_real'),
    scol('prediction').alias('salario_pred')
).show()

print(f'\nCoeficientes  : {lr_reg_model.coefficients}')
print(f'Intercepto    : {lr_reg_model.intercept:.2f}')

---
## Sección 3 — Aprendizaje No Supervisado (K-Means Clustering)

**Objetivo:** Segmentar a los empleados en grupos naturales sin usar el target `abandono`.  
Pregunta de negocio: *"¿Qué tipos de empleados tenemos según su perfil?"*

### 3.1 Aplicar K-Means con K=3

In [ ]:
from pyspark.ml.clustering import KMeans

# Para clustering usamos TODO el dataset limpio (no hay train/test — es no supervisado)
# Primero generamos features para el dataset completo
full_prep = prep_model.transform(df_clean)

In [ ]:
full_prep.show()

In [ ]:
kmeans = KMeans(
    featuresCol='features',
    predictionCol='cluster',
    k=3,
    maxIter=20,
    seed=42
)

In [ ]:
kmeans_model = kmeans.fit(full_prep)
df_clustered = kmeans_model.transform(full_prep)

In [ ]:
print('── Empleados y su cluster asignado ─────────────────')
df_clustered.select(
    'id', 'edad', 'salario', 'antiguedad', 'satisfaccion', 'dept', 'cluster'
).orderBy('cluster').show()

### 3.2 Método del Codo — ¿Cuál es el K óptimo?

In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

# WCSS = Within-Cluster Sum of Squares (suma de distancias al centroide)
# A menor WCSS, más compactos son los clusters
# El 'codo' es donde reducir K deja de mejorar significativamente el WCSS

wssse_results = []
silhouette_results = []

evaluator = ClusteringEvaluator(
    featuresCol='features',
    predictionCol='cluster',
    metricName='silhouette',
    distanceMeasure='squaredEuclidean'
)

for k in range(2, 7):
    km = KMeans(featuresCol='features', predictionCol='cluster', k=k, seed=42)
    m  = km.fit(full_prep)
    df_pred = m.transform(full_prep)
    
    wssse = m.summary.trainingCost          # WCSS
    silh  = evaluator.evaluate(df_pred)     # Silhouette [-1, 1], más alto es mejor
    
    wssse_results.append((k, wssse))
    silhouette_results.append((k, silh))
    print(f'K={k}  →  WCSS={wssse:10.2f}  |  Silhouette={silh:.4f}')


# Buscar el K donde WCSS baja menos entre K y K+1 (el 'codo'). 
# Silhouette > 0.5 es generalmente un buen clustering.

### 3.3 Interpretar los clusters

In [ ]:
# Perfil promedio de cada cluster — esto da la interpretación de negocio
print('── Perfil promedio por cluster ─────────────────────')
(
    df_clustered
    .groupBy('cluster')
    .agg(
        F.round(F.mean('edad'),          1).alias('edad_prom'),
        F.round(F.mean('salario'),       0).alias('salario_prom'),
        F.round(F.mean('antiguedad'),    1).alias('antig_prom'),
        F.round(F.mean('satisfaccion'),  2).alias('satisf_prom'),
        F.round(F.mean('abandono'),      2).alias('tasa_abandono'),
        F.count('id').alias('n_empleados')
    )
    .orderBy('cluster')
    .show()
)

# Centroides en el espacio de features escaladas
print('── Centroides de cada cluster (features escaladas) ─')
for i, c in enumerate(kmeans_model.clusterCenters()):
    print(f'Cluster {i}: {[round(x,3) for x in c]}')

---
## Sección 4 — Métricas de Rendimiento

**Objetivo:** Evaluar todos los modelos entrenados en las secciones 2 y 3 usando las métricas formales de Spark MLlib.  
Al final compararemos los dos clasificadores y el regresor en una tabla resumen.

### 4.1 Métricas de Regresión — MAE, MSE, RMSE

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

reg_eval = RegressionEvaluator(
    labelCol='label',
    predictionCol='prediction'
)

mae  = reg_eval.setMetricName('mae').evaluate(pred_reg)
mse  = reg_eval.setMetricName('mse').evaluate(pred_reg)
rmse = reg_eval.setMetricName('rmse').evaluate(pred_reg)
r2   = reg_eval.setMetricName('r2').evaluate(pred_reg)

print('── Métricas de Regresión Lineal (predicción de salario) ──')
print(f'  MAE   (Error Absoluto Medio)       : {mae:,.2f}')
print(f'  MSE   (Error Cuadrático Medio)     : {mse:,.2f}')
print(f'  RMSE  (Raíz del Error Cuadrático)  : {rmse:,.2f}')
print(f'  R²    (Coef. de determinación)     : {r2:.4f}')
print()
print('Interpretación:')
print(f'  En promedio, el modelo se equivoca ${mae:,.0f} en la predicción del salario.')
print(f'  R²={r2:.2f} significa que el modelo explica el {r2*100:.0f}% de la variabilidad del salario.')

### 4.2 Métricas de Clasificación — Confusion Matrix, Accuracy, Precision, Recall, F1

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

def evaluar_clasificador(nombre, predicciones):
    """Calcula y muestra todas las métricas de clasificación para un modelo."""
    
    eval_multi = MulticlassClassificationEvaluator(
        labelCol='label',
        predictionCol='prediction'
    )
    eval_bin = BinaryClassificationEvaluator(
        labelCol='label',
        rawPredictionCol='rawPrediction'
    )
    
    accuracy  = eval_multi.setMetricName('accuracy').evaluate(predicciones)
    precision = eval_multi.setMetricName('weightedPrecision').evaluate(predicciones)
    recall    = eval_multi.setMetricName('weightedRecall').evaluate(predicciones)
    f1        = eval_multi.setMetricName('f1').evaluate(predicciones)
    auc       = eval_bin.setMetricName('areaUnderROC').evaluate(predicciones)
    
    print(f'\n── {nombre} ─────────────────────────────────────────')
    print(f'  Accuracy  (exactitud)     : {accuracy:.4f}  → {accuracy*100:.1f}% clasificados correctamente')
    print(f'  Precision (precisión)     : {precision:.4f}')
    print(f'  Recall    (sensibilidad)  : {recall:.4f}')
    print(f'  F1-Score                  : {f1:.4f}')
    print(f'  AUC-ROC                   : {auc:.4f}  → {"Excelente" if auc>0.9 else "Bueno" if auc>0.7 else "Regular"}')
    
    # Confusion matrix manual
    print(f'\n  Confusion Matrix ({nombre}):')
    print(f'  {"":20} Predicho 0    Predicho 1')
    for real in [0.0, 1.0]:
        row = []
        for pred in [0.0, 1.0]:
            n = predicciones.filter(
                (scol('label') == real) & (scol('prediction') == pred)
            ).count()
            row.append(n)
        label_str = 'Real 0 (se queda) ' if real == 0.0 else 'Real 1 (abandona)'
        print(f'  {label_str}:  TP/TN={row[0]}          FP/FN={row[1]}')
    
    return {'modelo': nombre, 'accuracy': accuracy, 'precision': precision,
            'recall': recall, 'f1': f1, 'auc': auc}

resultados = []
resultados.append(evaluar_clasificador('Árbol de Decisión',      pred_dt))
resultados.append(evaluar_clasificador('Regresión Logística',    pred_lr))

### 4.3 Curva AUC-ROC

In [ ]:
# Calcular y comparar AUC de ambos modelos
from pyspark.ml.evaluation import BinaryClassificationEvaluator

bin_eval = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction'
)

print('── AUC-ROC por modelo ──────────────────────────────')
for nombre, preds in [('Árbol de Decisión', pred_dt), ('Regresión Logística', pred_lr)]:
    auc = bin_eval.setMetricName('areaUnderROC').evaluate(preds)
    pr  = bin_eval.setMetricName('areaUnderPR').evaluate(preds)
    
    # Representación visual del AUC en texto
    barra = '█' * int(auc * 20)
    print(f'\n  {nombre}')
    print(f'  AUC-ROC: {auc:.4f}  {barra}')
    print(f'  AUC-PR : {pr:.4f}  (Precision-Recall — más informativo con clases desbalanceadas)')


# AUC-PR es más informativo cuando las clases están desbalanceadas 
# (pocos abandonos vs muchos que se quedan — caso frecuente en RR.HH.) 

### 4.4 Tabla Resumen — Comparación de todos los modelos

In [ ]:
print('='*72)
print('RESUMEN COMPARATIVO DE MODELOS')
print('='*72)

print('\n📌 Modelos de CLASIFICACIÓN (target: abandono 0/1)\n')
print(f'{"Modelo":<25} {"Accuracy":>10} {"Precision":>10} {"Recall":>8} {"F1":>8} {"AUC":>8}')
print('-'*72)
for r in resultados:
    print(f"{r['modelo']:<25} {r['accuracy']:>10.4f} {r['precision']:>10.4f} "
          f"{r['recall']:>8.4f} {r['f1']:>8.4f} {r['auc']:>8.4f}")

print(f'\n📌 Modelo de REGRESIÓN (target: salario)\n')
print(f'{"Modelo":<25} {"MAE":>12} {"RMSE":>12} {"R²":>8}')
print('-'*55)
print(f'{"Regresión Lineal":<25} {mae:>12,.2f} {rmse:>12,.2f} {r2:>8.4f}')

print(f'\n📌 Clustering K-Means (K=3, no supervisado)\n')
km3 = KMeans(featuresCol='features', predictionCol='cluster', k=3, seed=42).fit(full_prep)
df_km3 = km3.transform(full_prep)
sil3 = ClusteringEvaluator(featuresCol='features', predictionCol='cluster').evaluate(df_km3)
print(f'  K=3  |  WCSS={km3.summary.trainingCost:,.2f}  |  Silhouette={sil3:.4f}')

print('\n' + '='*72)
print('Ciclo CRISP-DM completado: Datos → Limpieza → Features → Modelos → Evaluación')
print('='*72)

---
## Cierre

### Lo que hicimos en este notebook:

| Paso | Qué hicimos | Herramienta |
|---|---|---|
| Cargar datos | Parquet en HDFS | `spark.read.parquet` |
| Explorar | Describe, nulos, duplicados | DataFrame API |
| Limpiar | Outliers, nulos, duplicados | DataFrame API |
| Correlación | Pearson entre variables | `.stat.corr()` |
| Feature engineering | StringIndexer, VectorAssembler, StandardScaler | `pyspark.ml.feature` |
| Pipeline | Encadenar transformaciones | `pyspark.ml.Pipeline` |
| Clasificación | Árbol de Decisión, Regresión Logística | `pyspark.ml.classification` |
| Regresión | Regresión Lineal | `pyspark.ml.regression` |
| Clustering | K-Means, método del codo, Silhouette | `pyspark.ml.clustering` |
| Métricas | MAE/RMSE, Accuracy/F1/AUC-ROC, Silhouette | `pyspark.ml.evaluation` |

In [ ]:
# Siempre cerrar la SparkSession al terminar
spark.stop()
print('SparkSession cerrada correctamente.')